# 09 Capon-cycle term and diagonal-loading sensitivity

This notebook isolates `PhysicsComponents.capon_cycle`. The term builds a covariance from the predicted profile, applies diagonal loading, inverts it, forms the Capon spectrum, and compares its normalised shape against the target. Because of the Capon bias seen in notebook 04, even a self-comparison is not exactly zero; we characterise the residual and its dependence on the loading factor.

Modules exercised: `PhysicsComponents.capon_cycle`, `TomoGeometry`.

We show the term is small but nonzero for matched profiles, grows for mismatched profiles, and depends on the diagonal-loading regulariser.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
from configuration.training_config import GeometryConfig
from tools.sar.tomo_geometry            import TomoGeometry
from tools.data.gaussians                import GaussianMixture
from pipelines.backbone_pipeline.loss import PhysicsComponents

def to_check_tensor(profiles):
    t = torch.tensor(np.asarray(profiles, dtype=np.float32), device=DEVICE)
    return t.T.reshape(1, t.shape[1], 1, t.shape[0])

def gaussian_profile(x_axis, amp, mu, sigma):
    amps = np.asarray(amp,   dtype=np.float32).reshape(1, -1)
    mus  = np.asarray(mu,    dtype=np.float32).reshape(1, -1)
    sigs = np.asarray(sigma, dtype=np.float32).reshape(1, -1)
    return GaussianMixture.evaluate_batch(x_axis, amps, mus, sigs)[0]


In [ ]:
x_axis = np.linspace(-20.0, 40.0, 160).astype(np.float32)
dx     = float(x_axis[1] - x_axis[0])
xt     = torch.tensor(x_axis, device=DEVICE)
geom   = TomoGeometry(GeometryConfig(), xt)
floor  = 1e-3

def capon_cycle(pred, target, loading):
    return float(PhysicsComponents.capon_cycle(to_check_tensor(pred[None, :]), to_check_tensor(target[None, :]), geom.steering, geom.outer, dx, loading, floor))


## Matched versus mismatched profiles

At the default loading of `1e-2` the term is at the level of the Capon bias for a matched pair and rises for a mean-shifted or widened prediction.

In [ ]:
target  = gaussian_profile(x_axis, 1.0, 8.0, 4.0)
shifted = gaussian_profile(x_axis, 1.0, 16.0, 4.0)
widened = gaussian_profile(x_axis, 1.0, 8.0, 8.0)
loading = 1e-2

print(f"capon_cycle(target,  target) = {capon_cycle(target, target, loading):.6e}")
print(f"capon_cycle(shifted, target) = {capon_cycle(shifted, target, loading):.6e}")
print(f"capon_cycle(widened, target) = {capon_cycle(widened, target, loading):.6e}")

## Penalty across mean shift

We sweep the predicted mean and confirm the term grows away from the true mean, while noting the nonzero self-comparison floor set by the Capon bias.

In [ ]:
shifts = np.linspace(0.0, 20.0, 21)
vals   = [capon_cycle(gaussian_profile(x_axis, 1.0, 8.0 + s, 4.0), target, loading) for s in shifts]

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(shifts, vals, "o-", color="C3")
ax.axhline(capon_cycle(target, target, loading), color="0.4", ls="--", lw=0.9, label="self-comparison floor")
ax.set_xlabel("mean shift of predicted profile [m]")
ax.set_ylabel("capon_cycle term")
ax.set_title("Capon-cycle penalty vs mean shift")
ax.legend()
plt.show()

## Diagonal-loading sensitivity

The diagonal-loading factor regularises the covariance inversion. Too little loading lets the Capon estimator over-sharpen and amplifies the bias; more loading drives the estimator toward beamforming. We trace the matched-pair residual across loading values.

In [ ]:
loadings = np.geomspace(1e-4, 1e0, 25)
self_res = [capon_cycle(target, target, float(l)) for l in loadings]
mism_res = [capon_cycle(shifted, target, float(l)) for l in loadings]

fig, ax = plt.subplots(figsize=(6.6, 3.6))
ax.plot(loadings, self_res, "o-", color="C0", label="matched")
ax.plot(loadings, mism_res, "s-", color="C3", label="mean-shift")
ax.axvline(1e-2, color="0.4", ls="--", lw=0.9, label="default loading")
ax.set_xscale("log")
ax.set_xlabel("diagonal loading")
ax.set_ylabel("capon_cycle term")
ax.set_title("Capon-cycle residual vs diagonal loading")
ax.legend()
plt.show()

## Expected outcome

The matched-pair Capon-cycle term is small but strictly positive, reflecting the Capon bias rather than a profile mismatch. The penalty rises with mean shift above that floor. The loading sweep shows the residual is largest at very small loading and decreases as the estimator is regularised, explaining why the default loading is chosen well above the ill-conditioned regime.